# Plain Analytical Avellaneda-Stoikov on DOGE

This notebook computes daily metrics for a deliberately simple analytical Avellaneda-Stoikov market maker on the DOGE replay data.

Purpose: isolate the base A-S formula without gamma tuning, drawdown penalties, CVaR constraints, PPO, or Alpha-AS parameter learning. The only data-dependent part is estimating market parameters from the training period so the model can be run on the held-out DOGE days without test-day lookahead.

In [ ]:
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
     if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.data_loader import load_multi_day
from procs.gym.calibration import calibrate_from_arrays
from procs.gym.helpers.fast_rollout import fast_simulate

cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()
print(f"Repo root : {cfg.repo_root}")
print(f"Datasets  : {cfg.datasets_dir}")
print(f"Results   : {cfg.results_dir}")

## Metric Formulas

`fast_simulate` computes these metrics during the A-S rollout using the same streaming definitions used elsewhere in the project. The compact functions below are included only to make the definitions transparent on a toy path.

In [ ]:
def sharpe_from_pnl(pnl_path):
    returns = np.diff(np.asarray(pnl_path, dtype=float))
    if returns.size == 0 or returns.std() <= 1e-15:
        return 0.0
    return float(returns.mean() / returns.std())


def sortino_from_pnl(pnl_path):
    returns = np.diff(np.asarray(pnl_path, dtype=float))
    downside = returns[returns < 0]
    if downside.size < 2 or downside.std() <= 1e-15:
        return 0.0
    return float(returns.mean() / downside.std())


def max_drawdown_from_pnl(pnl_path):
    pnl = np.asarray(pnl_path, dtype=float)
    running_max = np.maximum.accumulate(pnl)
    return float(np.max(running_max - pnl))


def final_pnl(pnl_path):
    return float(np.asarray(pnl_path, dtype=float)[-1])


def mean_abs_inventory(inventory_path):
    return float(np.mean(np.abs(np.asarray(inventory_path, dtype=float))))


def pnl_to_map(pnl_path, inventory_path):
    mean_abs_q = mean_abs_inventory(inventory_path)
    return float(final_pnl(pnl_path) / mean_abs_q) if mean_abs_q > 1e-15 else 0.0

In [ ]:
toy_pnl = np.array([0.0, 1.0, 0.5, 1.5, 1.2, 2.0])
toy_inventory = np.array([0, 1, 1, 0, -1, 0])

pd.Series({
    "Sharpe": sharpe_from_pnl(toy_pnl),
    "Sortino": sortino_from_pnl(toy_pnl),
    "Max DD": max_drawdown_from_pnl(toy_pnl),
    "P&L-to-MAP": pnl_to_map(toy_pnl, toy_inventory),
    "Final PnL": final_pnl(toy_pnl),
    "Mean |q|": mean_abs_inventory(toy_inventory),
}, name="toy_day")

## Load DOGE Train/Test Days

The split mirrors the final notebooks: days 1-6 are used only to estimate fixed market parameters; days 7-29 are held out for reporting.

In [ ]:
TRAIN_DAYS = 6
EXPECTED_TEST_DAYS = 23

daily_S, daily_dt, dates = load_multi_day(str(cfg.datasets_dir), pair=cfg.pair)
train_S = daily_S[:TRAIN_DAYS]
train_dt = daily_dt[:TRAIN_DAYS]
train_dates = dates[:TRAIN_DAYS]
test_S = daily_S[TRAIN_DAYS:]
test_dt = daily_dt[TRAIN_DAYS:]
test_dates = dates[TRAIN_DAYS:]

if len(train_S) != TRAIN_DAYS or len(test_S) != EXPECTED_TEST_DAYS:
    raise ValueError(f"Expected {TRAIN_DAYS} train and {EXPECTED_TEST_DAYS} test days, found {len(train_S)} and {len(test_S)}.")
if set(map(str, train_dates)) & set(map(str, test_dates)):
    raise ValueError("Train/test date overlap detected.")

print(f"Training days: {train_dates[0]} -> {train_dates[-1]}")
print(f"Test days    : {test_dates[0]} -> {test_dates[-1]}")

## Fixed Parameters for the Plain A-S Model

This notebook intentionally does **not** tune `gamma`. We set a single plain A-S risk-aversion value and keep it fixed for all held-out days. `sigma`, `A`, and `kappa` are estimated from training days only and aggregated by median. This is not an improved or optimized baseline; it is a clean diagnostic for the raw analytical A-S formula on the DOGE data.

In [ ]:
PLAIN_AS_GAMMA = 0.1

train_params = []
for S, dt, date in zip(train_S, train_dt, train_dates):
    sigma_day, A_day, kappa_day = calibrate_from_arrays(S, dt, tick_size=cfg.tick_size)
    train_params.append({"date": str(date), "sigma": sigma_day, "A": A_day, "kappa": kappa_day})

train_param_frame = pd.DataFrame(train_params).set_index("date")
sigma_plain = float(train_param_frame["sigma"].median())
A_plain = float(train_param_frame["A"].median())
kappa_plain = float(train_param_frame["kappa"].median())

print("Plain A-S parameters used for every held-out day:")
print(f"  gamma = {PLAIN_AS_GAMMA:.6f}  (fixed, not tuned)")
print(f"  sigma = {sigma_plain:.6f}  (train-only median)")
print(f"  A     = {A_plain:.4f}  (train-only median)")
print(f"  kappa = {kappa_plain:.0f}  (train-only median)")
display(train_param_frame)

## Run Plain A-S on Held-Out Days

Each row is one held-out day. Fill randomness is averaged over `cfg.evaluation_rollouts` trajectories so the stochastic fill protocol matches the final notebooks.

In [ ]:
rows = []
for i, (S, dt, date) in enumerate(zip(test_S, test_dt, test_dates)):
    stats = fast_simulate(
        midprices=S,
        dt_array=dt,
        gamma=PLAIN_AS_GAMMA,
        sigma=sigma_plain,
        kappa=kappa_plain,
        A=A_plain,
        terminal_time=float(dt.sum()),
        tick_size=cfg.tick_size,
        Q_MAX=cfg.q_max,
        num_trajectories=cfg.evaluation_rollouts,
        seed=cfg.evaluation_seed,
        use_linear_approximation=False,
    )
    rows.append({
        "Day": str(date),
        "Sharpe": float(stats["sharpe"].mean()),
        "Sortino": float(stats["sortino"].mean()),
        "Max DD": float(stats["max_drawdown"].mean()),
        "P&L-to-MAP": float(stats["pnl_to_map"].mean()),
        "Final PnL": float(stats["total_pnl"].mean()),
        "Mean |q|": float(stats["mean_abs_q"].mean()),
        "Terminal q": float(stats["terminal_q"].mean()),
        "Mean spread": float(stats["mean_spread"].mean()),
        "Near Cap Fraction": float(stats.get("near_cap_fraction", np.zeros(cfg.evaluation_rollouts)).mean()),
        "Rollouts": float(cfg.evaluation_rollouts),
    })
    print(f"[{i+1:02d}/{len(test_S)}] {date}: Sharpe={rows[-1]['Sharpe']:.4f}, MaxDD={rows[-1]['Max DD']:.6f}, PnL={rows[-1]['Final PnL']:.6f}")

df_plain_as = pd.DataFrame(rows).set_index("Day")
df_plain_as

In [ ]:
summary = pd.DataFrame({
    "Mean": df_plain_as.mean(numeric_only=True),
    "Median": df_plain_as.median(numeric_only=True),
    "Std": df_plain_as.std(numeric_only=True),
})
summary.loc[["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL", "Mean |q|"]]

In [ ]:
out_path = cfg.result_path("plain_as_test_results.csv")
df_plain_as.to_csv(out_path)
print(f"Saved plain A-S daily metrics -> {out_path}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
plot_metrics = ["Sharpe", "Max DD", "Final PnL", "Mean |q|"]
for ax, metric in zip(axes.ravel(), plot_metrics):
    df_plain_as[metric].plot(ax=ax, marker="o", linewidth=1.2)
    ax.set_title(metric)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("Plain Analytical Avellaneda-Stoikov on DOGE Held-Out Days")
plt.tight_layout()
plt.show()

## Interpretation

This is the simplest analytical A-S diagnostic in this project. It is useful for understanding what the unoptimized formula does on the DOGE replay data. It should not replace the main B0 baseline unless the thesis question is specifically about an untuned analytical benchmark. For fair model comparison, use a consistent information protocol across all methods.